# 🚀 Titanic ML Pipeline - 데이터 준비

이 노트북은 로컬의 설정 파일과 데이터를 S3에 업로드하여 모델링 준비를 완료합니다.

## S3 구조
```
# 설정 버킷 (conf)
s3://gs-retail-awesome-conf-{region}/{env}/{user_id}/{project}/{experiment}/
    ├── env.yml
    ├── meta.yml
    └── model.yml

# 데이터 버킷 (data)
s3://gs-retail-awesome-data-{region}/{env}/{user_id}/{project}/{version}/
    └── data/
        ├── train.csv
        ├── validation.csv
        └── test.csv
```

## 1. 환경 설정

In [1]:
import os
import yaml
import boto3
from pathlib import Path
from datetime import datetime
from botocore.exceptions import ClientError

# 현재 디렉토리 확인
BASE_DIR = Path.cwd()
CONF_DIR = BASE_DIR / 'conf'
DATA_DIR = BASE_DIR / 'data'

print(f"📁 Base Directory: {BASE_DIR}")
print(f"📁 Conf Directory: {CONF_DIR}")
print(f"📁 Data Directory: {DATA_DIR}")

📁 Base Directory: /Users/kunops/GSRProject/998-Study-Apps/gs-ds-env/tests/titanic/notebooks/kunops/run_pm_2/modeling
📁 Conf Directory: /Users/kunops/GSRProject/998-Study-Apps/gs-ds-env/tests/titanic/notebooks/kunops/run_pm_2/modeling/conf
📁 Data Directory: /Users/kunops/GSRProject/998-Study-Apps/gs-ds-env/tests/titanic/notebooks/kunops/run_pm_2/modeling/data


## 2. 설정 파일 로드

In [2]:
def load_yaml(filepath: Path) -> dict:
    """YAML 파일 로드"""
    with open(filepath, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

# 설정 파일 로드
env_config = load_yaml(CONF_DIR / 'env.yml')
meta_config = load_yaml(CONF_DIR / 'meta.yml')
model_config = load_yaml(CONF_DIR / 'model.yml')

print("✅ 설정 파일 로드 완료")
print(f"   - env: {env_config['env']}")
print(f"   - region: {env_config['region']}")
print(f"   - project: {meta_config['user_id']}")
print(f"   - project: {meta_config['project']}")
print(f"   - experiment: {meta_config['experiment']}")
print(f"   - version: {meta_config['version']}")

✅ 설정 파일 로드 완료
   - env: dev
   - region: us-west-1
   - project: kunops
   - project: titanic-survival-prediction
   - experiment: kunops-v1
   - version: v1.0


## 3. S3 경로 생성

In [3]:
# 설정값 추출
ENV = env_config['env']
REGION = env_config['region']
USER_ID = meta_config['user_id']
PROJECT = meta_config['project']
EXPERIMENT = meta_config['experiment']
VERSION = meta_config['version']

# S3 버킷
CONF_BUCKET = env_config['s3']['conf_bucket']
DATA_BUCKET = env_config['s3']['data_bucket']
MODEL_BUCKET = env_config['s3']['model_bucket']

# 모든 버킷 목록
ALL_BUCKETS = [CONF_BUCKET, DATA_BUCKET, MODEL_BUCKET]

# S3 경로 (prefix)
CONF_PREFIX = f"{ENV}/{USER_ID}/{PROJECT}/{EXPERIMENT}"
DATA_PREFIX = f"{ENV}/{USER_ID}/{PROJECT}/{VERSION}"

# 전체 S3 URI
S3_CONF_PATH = f"s3://{CONF_BUCKET}/{CONF_PREFIX}/"
S3_DATA_PATH = f"s3://{DATA_BUCKET}/{DATA_PREFIX}/"

print("📦 S3 경로 구성")
print("")
print(f"   CONF: {S3_CONF_PATH}")
print(f"   DATA: {S3_DATA_PATH}")

📦 S3 경로 구성

   CONF: s3://gs-retail-awesome-conf-us-west-1/dev/kunops/titanic-survival-prediction/kunops-v1/
   DATA: s3://gs-retail-awesome-data-us-west-1/dev/kunops/titanic-survival-prediction/v1.0/


## 4. S3 클라이언트 초기화

In [4]:
os.environ["AWS_PROFILE"] = "155954279556_kunops"

# boto3 S3 클라이언트
s3_client = boto3.client('s3', region_name=REGION)

print(f"✅ S3 클라이언트 초기화 완료 (region: {REGION})")

✅ S3 클라이언트 초기화 완료 (region: us-west-1)


## 5. S3 버킷 확인 및 생성

In [5]:
def bucket_exists(bucket_name: str) -> bool:
    """S3 버킷 존재 여부 확인"""
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        return True
    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code == "404":
            return False
        elif error_code == "403":
            # 버킷은 존재하지만 접근 권한 없음
            print(f"   ⚠️  {bucket_name}: 접근 권한 없음 (버킷은 존재)")
            return True
        else:
            raise e


def create_bucket(bucket_name: str, region: str) -> bool:
    """S3 버킷 생성"""
    try:
        # us-east-1은 LocationConstraint를 지정하지 않음
        if region == "us-east-1":
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            s3_client.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region},
            )
        return True
    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code == "BucketAlreadyOwnedByYou":
            return True
        elif error_code == "BucketAlreadyExists":
            print(f"   ❌ {bucket_name}: 버킷명이 이미 다른 계정에서 사용 중")
            return False
        else:
            print(f"   ❌ {bucket_name}: 생성 실패 - {e}")
            return False


def ensure_bucket_exists(bucket_name: str, region: str) -> bool:
    """버킷이 없으면 생성"""
    if bucket_exists(bucket_name):
        print(f"   ✅ {bucket_name} (존재)")
        return True
    else:
        print(f"   🆕 {bucket_name} 생성 중...")
        success = create_bucket(bucket_name, region)
        if success:
            print(f"   ✅ {bucket_name} (생성 완료)")
        return success

In [6]:
print("🪣 S3 버킷 확인 및 생성")
print("=" * 60)

bucket_status = {}
for bucket in ALL_BUCKETS:
    bucket_status[bucket] = ensure_bucket_exists(bucket, REGION)

print("=" * 60)

# 결과 요약
all_ready = all(bucket_status.values())
if all_ready:
    print("\n✅ 모든 버킷 준비 완료!")
else:
    failed = [b for b, s in bucket_status.items() if not s]
    print(f"\n❌ 버킷 준비 실패: {failed}")
    print("   위 버킷을 수동으로 생성하거나 권한을 확인하세요.")

🪣 S3 버킷 확인 및 생성
   ✅ gs-retail-awesome-conf-us-west-1 (존재)
   ✅ gs-retail-awesome-data-us-west-1 (존재)
   ✅ gs-retail-awesome-model-us-west-1 (존재)

✅ 모든 버킷 준비 완료!


## 6. 헬퍼 함수 정의

In [7]:
def upload_file_to_s3(local_path: Path, bucket: str, s3_key: str) -> bool:
    """파일을 S3에 업로드"""
    try:
        s3_client.upload_file(str(local_path), bucket, s3_key)
        return True
    except Exception as e:
        print(f"❌ 업로드 실패: {local_path} -> {e}")
        return False


def check_s3_object_exists(bucket: str, key: str) -> bool:
    """S3 객체 존재 여부 확인"""
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except Exception as e:
        print(f"❌ 객체 존재 실패 -> {e}")
        return False


print("✅ 헬퍼 함수 정의 완료")

✅ 헬퍼 함수 정의 완료


## 7. 설정 파일 업로드 (conf 버킷)

In [8]:
# 업로드할 설정 파일 목록
CONF_FILES = ["env.yml", "meta.yml", "model.yml"]

print("📤 설정 파일 업로드 시작")
print(f"   대상: {S3_CONF_PATH}")
print("   " + "-" * 50)

conf_results = []
for filename in CONF_FILES:
    local_path = CONF_DIR / filename
    s3_key = f"{CONF_PREFIX}/{filename}"

    if local_path.exists():
        success = upload_file_to_s3(local_path, CONF_BUCKET, s3_key)
        status = "✅" if success else "❌"
        conf_results.append((filename, success))
        print(f"   {status} {filename} -> s3://{CONF_BUCKET}/{s3_key}")
    else:
        print(f"   ⚠️  {filename} 파일 없음 (스킵)")
        conf_results.append((filename, False))

print("   " + "-" * 50)
print(f"   완료: {sum(1 for _, s in conf_results if s)}/{len(CONF_FILES)}")

📤 설정 파일 업로드 시작
   대상: s3://gs-retail-awesome-conf-us-west-1/dev/kunops/titanic-survival-prediction/kunops-v1/
   --------------------------------------------------
   ✅ env.yml -> s3://gs-retail-awesome-conf-us-west-1/dev/kunops/titanic-survival-prediction/kunops-v1/env.yml
   ✅ meta.yml -> s3://gs-retail-awesome-conf-us-west-1/dev/kunops/titanic-survival-prediction/kunops-v1/meta.yml
   ✅ model.yml -> s3://gs-retail-awesome-conf-us-west-1/dev/kunops/titanic-survival-prediction/kunops-v1/model.yml
   --------------------------------------------------
   완료: 3/3


## 7-1. 노트북 파일 업로드 (conf 버킷)

`meta.yml`의 `notebook` 필드에 지정된 ipynb 파일을 conf 버킷에 함께 업로드합니다.
`run_pm.py`가 이 노트북을 다운로드하여 papermill로 실행합니다.

In [15]:
# 업로드할 노트북 파일 목록
NOTEBOOK_FILES = ["titanic_modeling.ipynb", "titanic_modeling_mlflow.ipynb"]

print("📤 노트북 파일 업로드")
print("   " + "-" * 50)

for nb_name in NOTEBOOK_FILES:
    nb_path = BASE_DIR / nb_name
    s3_key = f"{CONF_PREFIX}/{nb_name}"

    if nb_path.exists():
        file_size = nb_path.stat().st_size
        size_str = f"{file_size / 1024:.1f} KB" if file_size < 1024 * 1024 else f"{file_size / (1024*1024):.1f} MB"

        success = upload_file_to_s3(nb_path, CONF_BUCKET, s3_key)
        status = "✅" if success else "❌"
        print(f"   {status} {nb_name} ({size_str}) -> s3://{CONF_BUCKET}/{s3_key}")
    else:
        print(f"   ❌ {nb_name} 파일을 찾을 수 없습니다: {nb_path}")

print("   " + "-" * 50)
print(f"   완료: {len(NOTEBOOK_FILES)} 노트북")

📤 노트북 파일 업로드
   --------------------------------------------------
   ✅ titanic_modeling.ipynb (37.6 KB) -> s3://gs-retail-awesome-conf-us-west-1/dev/kunops/titanic-survival-prediction/kunops-v1/titanic_modeling.ipynb
   ✅ titanic_modeling_mlflow.ipynb (40.4 KB) -> s3://gs-retail-awesome-conf-us-west-1/dev/kunops/titanic-survival-prediction/kunops-v1/titanic_modeling_mlflow.ipynb
   --------------------------------------------------
   완료: 2 노트북


## 8. 데이터 파일 업로드 (data 버킷)

In [10]:
# 업로드할 데이터 파일 목록
DATA_FILES = ["train.csv", "validation.csv", "test.csv"]

print("📤 데이터 파일 업로드 시작")
print(f"   대상: {S3_DATA_PATH}data/")
print("   " + "-" * 50)

data_results = []
for filename in DATA_FILES:
    local_path = DATA_DIR / filename
    s3_key = f"{DATA_PREFIX}/data/{filename}"

    if local_path.exists():
        # 파일 크기 확인
        file_size = local_path.stat().st_size
        size_str = (
            f"{file_size / 1024:.1f} KB"
            if file_size < 1024 * 1024
            else f"{file_size / (1024*1024):.1f} MB"
        )

        success = upload_file_to_s3(local_path, DATA_BUCKET, s3_key)
        status = "✅" if success else "❌"
        data_results.append((filename, success, file_size))
        print(f"   {status} {filename} ({size_str}) -> s3://{DATA_BUCKET}/{s3_key}")
    else:
        print(f"   ⚠️  {filename} 파일 없음 (스킵)")
        data_results.append((filename, False, 0))

print("   " + "-" * 50)
print(f"   완료: {sum(1 for _, s, _ in data_results if s)}/{len(DATA_FILES)}")

📤 데이터 파일 업로드 시작
   대상: s3://gs-retail-awesome-data-us-west-1/dev/kunops/titanic-survival-prediction/v1.0/data/
   --------------------------------------------------
   ✅ train.csv (59.8 KB) -> s3://gs-retail-awesome-data-us-west-1/dev/kunops/titanic-survival-prediction/v1.0/data/train.csv
   ✅ validation.csv (28.0 KB) -> s3://gs-retail-awesome-data-us-west-1/dev/kunops/titanic-survival-prediction/v1.0/data/validation.csv
   ✅ test.csv (28.0 KB) -> s3://gs-retail-awesome-data-us-west-1/dev/kunops/titanic-survival-prediction/v1.0/data/test.csv
   --------------------------------------------------
   완료: 3/3


## 9. 업로드 결과 검증

In [11]:
print("🔍 S3 업로드 검증")
print("=" * 60)

# Conf 버킷 검증
print(f"\n📦 Conf 버킷: {CONF_BUCKET}")
print(f"   Prefix: {CONF_PREFIX}/")
for filename in CONF_FILES:
    s3_key = f"{CONF_PREFIX}/{filename}"
    exists = check_s3_object_exists(CONF_BUCKET, s3_key)
    status = "✅" if exists else "❌"
    print(f"   {status} {filename}")

# Data 버킷 검증
print(f"\n📦 Data 버킷: {DATA_BUCKET}")
print(f"   Prefix: {DATA_PREFIX}/data/")
for filename in DATA_FILES:
    s3_key = f"{DATA_PREFIX}/data/{filename}"
    exists = check_s3_object_exists(DATA_BUCKET, s3_key)
    status = "✅" if exists else "❌"
    print(f"   {status} {filename}")

print("\n" + "=" * 60)

🔍 S3 업로드 검증

📦 Conf 버킷: gs-retail-awesome-conf-us-west-1
   Prefix: dev/kunops/titanic-survival-prediction/kunops-v1/
   ✅ env.yml
   ✅ meta.yml
   ✅ model.yml

📦 Data 버킷: gs-retail-awesome-data-us-west-1
   Prefix: dev/kunops/titanic-survival-prediction/v1.0/data/
   ✅ train.csv
   ✅ validation.csv
   ✅ test.csv



## 10. 최종 요약

In [12]:
print("\n" + "=" * 60)
print("📋 데이터 준비 완료 요약")
print("=" * 60)

print(
    f"""
🏷️  프로젝트 정보
    - Project:    {PROJECT}
    - Experiment: {EXPERIMENT}
    - Version:    {VERSION}
    - User:       {USER_ID}
    - Env:        {ENV}

🪣 S3 버킷
    - Conf:  {CONF_BUCKET}
    - Data:  {DATA_BUCKET}
    - Model: {MODEL_BUCKET}

📦 S3 경로
    - Conf: {S3_CONF_PATH}
    - Data: {S3_DATA_PATH}

📄 업로드된 파일
    [Conf] env.yml, meta.yml, model.yml
    [Data] train.csv, validation.csv, test.csv

🚀 다음 단계
    1. 학습 노트북에서 위 S3 경로 참조
    2. SageMaker Training Job 실행
    3. 결과물은 model 버킷에 자동 저장:
       s3://{MODEL_BUCKET}/{ENV}/{USER_ID}/{PROJECT}/{EXPERIMENT}/{{run_id}}/
"""
)

print("=" * 60)
print(f"⏰ 완료 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


📋 데이터 준비 완료 요약

🏷️  프로젝트 정보
    - Project:    titanic-survival-prediction
    - Experiment: kunops-v1
    - Version:    v1.0
    - User:       kunops
    - Env:        dev

🪣 S3 버킷
    - Conf:  gs-retail-awesome-conf-us-west-1
    - Data:  gs-retail-awesome-data-us-west-1
    - Model: gs-retail-awesome-model-us-west-1

📦 S3 경로
    - Conf: s3://gs-retail-awesome-conf-us-west-1/dev/kunops/titanic-survival-prediction/kunops-v1/
    - Data: s3://gs-retail-awesome-data-us-west-1/dev/kunops/titanic-survival-prediction/v1.0/

📄 업로드된 파일
    [Conf] env.yml, meta.yml, model.yml
    [Data] train.csv, validation.csv, test.csv

🚀 다음 단계
    1. 학습 노트북에서 위 S3 경로 참조
    2. SageMaker Training Job 실행
    3. 결과물은 model 버킷에 자동 저장:
       s3://gs-retail-awesome-model-us-west-1/dev/kunops/titanic-survival-prediction/kunops-v1/{run_id}/

⏰ 완료 시각: 2026-03-18 10:04:09


---

## 🔧 유틸리티 함수 (선택)

In [ ]:
def list_s3_objects(bucket: str, prefix: str, max_keys: int = 100) -> list:
    """S3 prefix 아래 객체 목록 조회"""
    response = s3_client.list_objects_v2(
        Bucket=bucket,
        Prefix=prefix,
        MaxKeys=max_keys
    )
    
    objects = []
    if 'Contents' in response:
        for obj in response['Contents']:
            objects.append({
                'key': obj['Key'],
                'size': obj['Size'],
                'last_modified': obj['LastModified']
            })
    return objects

# Conf 버킷 내용 확인
print(f"📂 Conf 버킷 내용:")
for obj in list_s3_objects(CONF_BUCKET, CONF_PREFIX):
    print(f"   - {obj['key']} ({obj['size']} bytes)")

print(f"\n📂 Data 버킷 내용:")
for obj in list_s3_objects(DATA_BUCKET, DATA_PREFIX):
    print(f"   - {obj['key']} ({obj['size']} bytes)")

In [ ]:
# AWS CLI 명령어 생성 (수동 확인용)
print("🖥️  AWS CLI 명령어 (수동 확인용)")
print("=" * 60)
print(f"\n# 버킷 목록 확인")
print(f"aws s3 ls | grep gs-retail-awesome")
print(f"\n# Conf 버킷 확인")
print(f"aws s3 ls {S3_CONF_PATH}")
print(f"\n# Data 버킷 확인")
print(f"aws s3 ls {S3_DATA_PATH} --recursive")
print(f"\n# 설정 파일 다운로드 (로컬 테스트용)")
print(f"aws s3 cp {S3_CONF_PATH} ./downloaded_conf/ --recursive")

---

## 🗑️ 정리 (선택 - 주의해서 사용)

In [ ]:
# ⚠️ 주의: 아래 코드는 버킷 내 데이터를 삭제합니다!
# 필요한 경우에만 주석 해제 후 실행하세요.

# def delete_s3_prefix(bucket: str, prefix: str):
#     """S3 prefix 아래 모든 객체 삭제"""
#     objects = list_s3_objects(bucket, prefix, max_keys=1000)
#     if objects:
#         delete_keys = [{'Key': obj['key']} for obj in objects]
#         s3_client.delete_objects(
#             Bucket=bucket,
#             Delete={'Objects': delete_keys}
#         )
#         print(f"   🗑️  {len(delete_keys)} 객체 삭제 완료")
#     else:
#         print(f"   ℹ️  삭제할 객체 없음")

# # Conf prefix 삭제
# print(f"Conf prefix 삭제: {CONF_PREFIX}")
# delete_s3_prefix(CONF_BUCKET, CONF_PREFIX)

# # Data prefix 삭제
# print(f"Data prefix 삭제: {DATA_PREFIX}")
# delete_s3_prefix(DATA_BUCKET, DATA_PREFIX)